In [1]:
import pandas as pd
import sqlite3

# Load the cleaned data
df = pd.read_csv('../data/processed/online_retail_cleaned.csv')
print(df.shape)

# Create a SQLite database and load the data into a table
conn = sqlite3.connect('../data/processed/retail.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

print("Loaded into SQLite.")

(805549, 8)
Loaded into SQLite.


In [2]:
query = """
SELECT COUNT(*) AS total_rows
FROM transactions;
"""
pd.read_sql_query(query, conn)

,total_rows
0,805549


In [3]:
query = """
SELECT 
    ROUND(SUM(Quantity * Price), 2) AS total_revenue,
    MIN(InvoiceDate) AS first_transaction,
    MAX(InvoiceDate) AS last_transaction,
    COUNT(DISTINCT Invoice) AS total_orders,
    COUNT(DISTINCT "Customer ID") AS total_customers
FROM transactions;
"""
pd.read_sql_query(query, conn)

,total_revenue,first_transaction,last_transaction,total_orders,total_customers
0,17743429.18,2009-12-01 07:45:00,2011-12-09 12:50:00,36969,5878


In [4]:
query = """
SELECT 
    strftime('%Y-%m', InvoiceDate) AS month,
    ROUND(SUM(Quantity * Price), 2) AS monthly_revenue,
    COUNT(DISTINCT Invoice) AS orders
FROM transactions
GROUP BY month
ORDER BY month;
"""
monthly_revenue = pd.read_sql_query(query, conn)
monthly_revenue

,month,monthly_revenue,orders
0,2009-12,686654.16,1512
1,2010-01,557319.06,1011
2,2010-02,506371.07,1104
3,2010-03,699608.99,1524
4,2010-04,594609.19,1329
5,2010-05,599985.79,1377
6,2010-06,639066.58,1497
7,2010-07,591636.74,1381
8,2010-08,604242.65,1293
9,2010-09,831615.00,1689


In [5]:
query = """
SELECT 
    "Customer ID",
    Country,
    ROUND(SUM(Quantity * Price), 2) AS total_spent,
    COUNT(DISTINCT Invoice) AS num_orders
FROM transactions
GROUP BY "Customer ID"
ORDER BY total_spent DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,Customer ID,Country,total_spent,num_orders
0,18102.0,United Kingdom,608821.65,145
1,14646.0,Netherlands,528602.52,151
2,14156.0,EIRE,313946.37,156
3,14911.0,EIRE,295972.63,398
4,17450.0,United Kingdom,246973.09,51
5,13694.0,United Kingdom,196482.81,143
6,17511.0,United Kingdom,175603.55,60
7,16446.0,United Kingdom,168472.50,2
8,16684.0,United Kingdom,147142.77,55
9,12415.0,Australia,144458.37,28


In [6]:
query = """
SELECT 
    StockCode,
    Description,
    ROUND(SUM(Quantity * Price), 2) AS total_revenue,
    SUM(Quantity) AS total_units_sold
FROM transactions
GROUP BY StockCode, Description
ORDER BY total_revenue DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,StockCode,Description,total_revenue,total_units_sold
0,22423,REGENCY CAKESTAND 3 TIER,286486.30,24899
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,252072.46,93640
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
3,M,Manual,152340.57,9803
4,85099B,JUMBO BAG RED RETROSPOT,136980.08,75759
5,84879,ASSORTED COLOUR BIRD ORNAMENT,127074.17,79913
6,POST,POSTAGE,126563.04,5333
7,47566,PARTY BUNTING,103880.23,23607
8,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916
9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,79594.33,29477


In [7]:
query = """
SELECT 
    Country,
    ROUND(SUM(Quantity * Price), 2) AS total_revenue,
    COUNT(DISTINCT "Customer ID") AS num_customers
FROM transactions
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,Country,total_revenue,num_customers
0,United Kingdom,14723147.52,5350
1,EIRE,621631.11,5
2,Netherlands,554232.34,22
3,Germany,431262.46,107
4,France,355257.47,95
5,Australia,169968.11,15
6,Spain,109178.53,41
7,Switzerland,100365.34,22
8,Sweden,91549.72,19
9,Denmark,69862.19,12


In [8]:
query = """
SELECT * FROM transactions
WHERE "Customer ID" = 16446.0
ORDER BY InvoiceDate;
"""
pd.read_sql_query(query, conn)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,553573,22980,PANTRY SCRUBBING BRUSH,1,2011-05-18 09:52:00,1.65,16446.0,United Kingdom
1,553573,22982,PANTRY PASTRY BRUSH,1,2011-05-18 09:52:00,1.25,16446.0,United Kingdom
2,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom


In [9]:
query = """
SELECT 
    StockCode,
    Description,
    ROUND(SUM(Quantity * Price), 2) AS total_revenue,
    SUM(Quantity) AS total_units_sold
FROM transactions
WHERE StockCode NOT IN ('M', 'POST', 'D', 'C2', 'DOT')
GROUP BY StockCode, Description
ORDER BY total_revenue DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,StockCode,Description,total_revenue,total_units_sold
0,22423,REGENCY CAKESTAND 3 TIER,286486.30,24899
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,252072.46,93640
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
3,85099B,JUMBO BAG RED RETROSPOT,136980.08,75759
4,84879,ASSORTED COLOUR BIRD ORNAMENT,127074.17,79913
5,47566,PARTY BUNTING,103880.23,23607
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,79594.33,29477
8,79321,CHILLI LIGHTS,72860.14,15735
9,21137,BLACK RECORD COVER FRAME,67209.44,19629


In [10]:
final_queries = """
-- ============================================
-- E-commerce Customer Churn & RFM Analysis
-- SQL Queries: Business Metrics Extraction
-- ============================================

-- 1. Overall business summary
SELECT 
    ROUND(SUM(Quantity * Price), 2) AS total_revenue,
    MIN(InvoiceDate) AS first_transaction,
    MAX(InvoiceDate) AS last_transaction,
    COUNT(DISTINCT Invoice) AS total_orders,
    COUNT(DISTINCT "Customer ID") AS total_customers
FROM transactions;

-- 2. Monthly revenue trend (note: Dec 2011 is a partial month, data ends Dec 9)
SELECT 
    strftime('%Y-%m', InvoiceDate) AS month,
    ROUND(SUM(Quantity * Price), 2) AS monthly_revenue,
    COUNT(DISTINCT Invoice) AS orders
FROM transactions
GROUP BY month
ORDER BY month;

-- 3. Top 10 customers by revenue
SELECT 
    "Customer ID",
    Country,
    ROUND(SUM(Quantity * Price), 2) AS total_spent,
    COUNT(DISTINCT Invoice) AS num_orders
FROM transactions
GROUP BY "Customer ID"
ORDER BY total_spent DESC
LIMIT 10;

-- 4. Top 10 products by revenue (excluding non-product codes: Manual, Postage, Discount, Carriage)
SELECT 
    StockCode,
    Description,
    ROUND(SUM(Quantity * Price), 2) AS total_revenue,
    SUM(Quantity) AS total_units_sold
FROM transactions
WHERE StockCode NOT IN ('M', 'POST', 'D', 'C2', 'DOT')
GROUP BY StockCode, Description
ORDER BY total_revenue DESC
LIMIT 10;

-- 5. Revenue by country
SELECT 
    Country,
    ROUND(SUM(Quantity * Price), 2) AS total_revenue,
    COUNT(DISTINCT "Customer ID") AS num_customers
FROM transactions
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 10;
"""

with open('../sql/queries.sql', 'w') as f:
    f.write(final_queries)

print("Saved sql/queries.sql")

Saved sql/queries.sql
